# 🤖 LLM Scoring (Typhoon 2.5)

**If NOT news → ALL scores = 0**

**Keyword Columns (regex-based):**
| Column | Checks for |
|--------|------------|
| A.I. | AI, A.I., artificial intelligence |
| Google | Google |
| Microsoft | Microsoft |
| Nvidia | Nvidia |
| Crypto | Crypto, Bitcoin, Ethereum |
| Chipset | Chipset, chip, semiconductor |

**LLM-Scored Features:**
| Feature | Description |
|---------|-------------|
| Social_Impact | Does this affect society? |
| Economic_Impact | Does this affect markets/economy? |
| No_Ads | Is this real news (not ad/listicle)? |
| Has_Reference | Does it cite sources/data/research? |

**Output:** `scored_news.csv`

In [36]:
# CELL 1: IMPORTS & CONFIG
import os
import json
import re
import pandas as pd
from datetime import datetime
from tqdm.notebook import tqdm

try:
    from llama_cpp import Llama
    print("✅ llama-cpp-python loaded")
except ImportError:
    print("❌ Install: pip install llama-cpp-python")
    raise

class CFG:
    INPUT_CSV = "scraped_news_v10.csv"  # or scraped_news_v10_full.csv
    OUTPUT_CSV = "scored_news.csv"
    MODEL_PATH = "../data/models/typhoon2.5-qwen3-4b-q4_k_m.gguf"
    
    TEST_LIMIT = 15  # Set to None for all
    
    N_CTX = 4096
    N_GPU_LAYERS = -1
    TEMPERATURE = 0.1
    MAX_TOKENS = 200
    
    KEYWORDS = ["A.I.", "Google", "Microsoft", "Nvidia", "Crypto", "Chipset"]
    LLM_FEATURES = ["Social_Impact", "Economic_Impact", "No_Ads", "Has_Reference"]

print(f"📂 Input: {CFG.INPUT_CSV}")
print(f"📂 Output: {CFG.OUTPUT_CSV}")
print(f"🧪 Test limit: {CFG.TEST_LIMIT}")

✅ llama-cpp-python loaded
📂 Input: scraped_news_v10.csv
📂 Output: scored_news.csv
🧪 Test limit: 15


In [37]:
# CELL 2: KEYWORD DETECTION
def detect_keywords(text):
    if not text:
        return {kw: 0 for kw in CFG.KEYWORDS}
    
    text_lower = text.lower()
    results = {}
    
    for kw in CFG.KEYWORDS:
        kw_lower = kw.lower()
        if kw_lower == "a.i.":
            if "a.i." in text_lower or re.search(r'\bai\b', text_lower):
                results[kw] = 1
            else:
                results[kw] = 0
        else:
            results[kw] = 1 if kw_lower in text_lower else 0
    
    return results

print("✅ Keyword detection ready")

✅ Keyword detection ready


In [38]:
# CELL 3: LOAD MODEL
print("Loading Typhoon 2.5...")

llm = Llama(
    model_path=CFG.MODEL_PATH,
    n_ctx=CFG.N_CTX,
    n_gpu_layers=CFG.N_GPU_LAYERS,
    verbose=False
)

print("✅ Model loaded!")

Loading Typhoon 2.5...


llama_context: n_ctx_per_seq (4096) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


✅ Model loaded!


In [39]:
# CELL 4: LLM SCORING PROMPT
# CRITICAL: If NOT news, ALL scores must be 0
SCORING_PROMPT = """You are a news analyst. First determine if this is a NEWS ARTICLE. Then score it.

ARTICLE:
Title: {title}
Content: {content}

STEP 1: Is this a NEWS ARTICLE?
NOT news if it is:
- Product deal/discount post ("Save $100", "deal:", "under $100")
- Gift guide or shopping list
- Product review or listicle ("Best 10 vacuums")
- Movie/game/TV show review
- Sponsored content or advertisement
- How-to guide or tutorial

If NOT news → Return ALL ZEROS: {{"Social_Impact": 0, "Economic_Impact": 0, "No_Ads": 0, "Has_Reference": 0}}

STEP 2: If it IS news, score these features:

1. Social_Impact:
   - 1 if: affects public policy, privacy, jobs, education, health, environment, or daily life
   - 0 if: only affects businesses, investors, or niche technical audience

2. Economic_Impact:
   - 1 if mentions: funding round, IPO, stock price, revenue, layoffs, acquisition, valuation, earnings
   - 0 if: just product announcement or technical news

3. No_Ads:
   - 1 if: genuine journalism (NOT a deal, listicle, review, or ad)
   - 0 if: product deal, gift guide, listicle, sponsored content

4. Has_Reference:
   - 1 if: cites specific numbers, quotes from named people, research, or official sources
   - 0 if: vague claims, no data, no named sources

Respond ONLY with JSON:
{{"Social_Impact": 0, "Economic_Impact": 0, "No_Ads": 1, "Has_Reference": 0}}

JSON:"""

print("✅ Prompt ready (NOT news = ALL zeros)")

✅ Prompt ready (NOT news = ALL zeros)


In [40]:
# CELL 5: LLM SCORING FUNCTION
def score_with_llm(title: str, content: str) -> dict:
    content_truncated = content[:2000] if content else ""
    prompt = SCORING_PROMPT.format(title=title, content=content_truncated)
    
    try:
        response = llm(
            prompt,
            max_tokens=CFG.MAX_TOKENS,
            temperature=CFG.TEMPERATURE,
            stop=["}", "\n\n"],
            echo=False
        )
        
        text = response['choices'][0]['text'].strip()
        if not text.endswith('}'):
            text = text + '}'
        
        scores = json.loads(text)
        
        result = {}
        for feature in CFG.LLM_FEATURES:
            val = scores.get(feature, 0)
            result[feature] = 1 if val == 1 or val == "1" or val == True else 0
        
        return result
        
    except:
        return {f: 0 for f in CFG.LLM_FEATURES}

print("✅ Scoring function ready")

✅ Scoring function ready


In [41]:
# CELL 6: PROCESS ARTICLES
df_full = pd.read_csv(CFG.INPUT_CSV)
print(f"📰 Loaded {len(df_full)} articles")

if CFG.TEST_LIMIT:
    df = df_full.head(CFG.TEST_LIMIT)
    print(f"🧪 Testing first {len(df)}")
else:
    df = df_full

results = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Scoring"):
    title = row.get('headline', '') or ''
    content = row.get('full_content', '') or row.get('content_snippet', '') or ''
    full_text = f"{title} {content}"
    
    keyword_scores = detect_keywords(full_text)
    llm_scores = score_with_llm(title, content)
    
    result = {
        'news_id': idx,
        'url_hash': row.get('url_hash', ''),
        'headline': title[:100],
        **keyword_scores,
        **llm_scores
    }
    results.append(result)

print(f"\n✅ Processed {len(results)} articles")

📰 Loaded 348 articles
🧪 Testing first 15


Scoring:   0%|          | 0/15 [00:00<?, ?it/s]


✅ Processed 15 articles


In [42]:
# CELL 7: SAVE RESULTS
scored_df = pd.DataFrame(results)
columns = ['news_id', 'url_hash', 'headline'] + CFG.KEYWORDS + CFG.LLM_FEATURES
scored_df = scored_df[columns]
scored_df.to_csv(CFG.OUTPUT_CSV, index=False)
print(f"💾 Saved to {CFG.OUTPUT_CSV}")
scored_df.head(10)

💾 Saved to scored_news.csv


,news_id,url_hash,headline,A.I.,Google,Microsoft,Nvidia,Crypto,Chipset,Social_Impact,Economic_Impact,No_Ads,Has_Reference
0,0,231ac019b44b645d0fbefe32d856de73,"Zencoder drops Zenflow, a free AI orchestratio...",1,1,1,0,0,0,0,0,1,0
1,1,074a456b8a72ebeedd196f84a160ce38,Apple TV adds support for Google Cast even as ...,0,1,0,0,0,0,0,0,1,0
2,2,a175e9242218816aad54c07f73622048,Echo raises $35M to secure the enterprise clou...,1,0,0,0,0,0,0,0,1,0
3,3,6a68b01f30bbf9c2c75b8a1cfe179a5e,Is ChatGPT’s New Shopping Research Solving a P...,1,0,0,0,0,0,0,0,1,0
4,4,4dc43cae86fec91848c48af20e5906cb,"With 91% accuracy, open source Hindsight agent...",1,0,0,0,0,0,0,0,1,0
5,5,9485a7722e3c77c74cdb1e96f21e4721,Zoom says it aced AI’s hardest exam. Critics s...,1,1,1,0,0,0,0,0,1,0
6,6,6b24ab1f94a449ce726f97d4aaeb906e,Databricks raises $4B at $134B valuation as it...,1,1,1,0,0,0,0,1,1,1
7,7,ce093c7dea85ec4ae0d0542778d2d766,Black box AI isn’t enough: Why enterprise cons...,1,0,0,0,0,0,0,0,1,0
8,8,c4cbf5014763841709c12bee24bceffb,Adobe Firefly now supports prompt-based video ...,1,1,1,0,0,0,0,0,1,0
9,9,f70df2cb6b9740cf349c3c9cf762a223,Tekpon acquires TNW (The Next Web) brand from ...,1,0,0,0,0,0,0,0,1,0


In [43]:
# CELL 8: ANALYTICS
print("📊 Scoring Summary")
print("="*60)

# Count non-news (all zeros)
non_news = scored_df[(scored_df['No_Ads'] == 0)]
real_news = scored_df[(scored_df['No_Ads'] == 1)]
print(f"\n📰 Real News: {len(real_news)} ({len(real_news)/len(scored_df)*100:.1f}%)")
print(f"🗑️ Not News (deals/reviews/etc): {len(non_news)} ({len(non_news)/len(scored_df)*100:.1f}%)")

print("\n🔑 KEYWORD COUNTS (real news only):")
for kw in CFG.KEYWORDS:
    count = real_news[kw].sum()
    pct = (count / len(real_news)) * 100 if len(real_news) > 0 else 0
    print(f"   {kw:12} {count:4} articles ({pct:5.1f}%)")

print("\n🤖 LLM FEATURES (real news only):")
for feature in CFG.LLM_FEATURES:
    count = real_news[feature].sum()
    pct = (count / len(real_news)) * 100 if len(real_news) > 0 else 0
    print(f"   {feature:20} {count:4} articles ({pct:5.1f}%)")

📊 Scoring Summary

📰 Real News: 15 (100.0%)
🗑️ Not News (deals/reviews/etc): 0 (0.0%)

🔑 KEYWORD COUNTS (real news only):
   A.I.           12 articles ( 80.0%)
   Google          8 articles ( 53.3%)
   Microsoft       6 articles ( 40.0%)
   Nvidia          1 articles (  6.7%)
   Crypto          2 articles ( 13.3%)
   Chipset         0 articles (  0.0%)

🤖 LLM FEATURES (real news only):
   Social_Impact           2 articles ( 13.3%)
   Economic_Impact         2 articles ( 13.3%)
   No_Ads                 15 articles (100.0%)
   Has_Reference           2 articles ( 13.3%)
